# Klasifikasi Tutupan Lahan Jawa Timur — Random Forest

Notebook lokal untuk membaca 350 sampel GeoJSON, mengekstrak enam band Sentinel-2 hasil clipping, melatih Random Forest, lalu menghasilkan peta interaktif Folium.

Letakkan notebook di folder proyek yang sama dengan `jawa_timur.geojson`, `lokasi.geojson` (opsional), dan folder `hasil_clip/` berisi `B02_clip.tif`, `B03_clip.tif`, `B04_clip.tif`, `B08_clip.tif`, `B8A_clip.tif`, `B11_clip.tif`.


## 1. Import pustaka dan atur lokasi file

Cell ini memuat pustaka untuk membaca GeoJSON dan raster, membangun model Random Forest, mengevaluasi hasil, serta membuat peta interaktif.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
import folium
from folium.raster_layers import ImageOverlay
warnings.filterwarnings("ignore", category=UserWarning)

BASE = Path.cwd()
SAMPLE_FILE = BASE / "jawa_timur.geojson"
RASTER_DIR = BASE / "hasil_clip"
BOUNDARY_FILE = BASE / "lokasi.geojson"
BANDS = ["B02", "B03", "B04", "B08", "B8A", "B11"]
RASTER_FILES = {b: RASTER_DIR / f"{b}_clip.tif" for b in BANDS}

print("Folder kerja:", BASE)
print("Sampel ditemukan:", SAMPLE_FILE.exists())
for b, p in RASTER_FILES.items():
    print(b, p.exists(), p)


## 2. Membaca data sampel

Baca `jawa_timur.geojson`, periksa jumlah fitur, CRS, jenis geometri, kolom atribut, dan nilai kolom yang berpotensi menjadi label kelas.

In [ ]:
# Baca sampel dan deteksi kolom label
if not SAMPLE_FILE.exists():
    raise FileNotFoundError(f"Tidak menemukan {SAMPLE_FILE}. Letakkan jawa_timur.geojson di folder notebook.")

samples = gpd.read_file(SAMPLE_FILE)
print("Jumlah fitur:", len(samples))
print("CRS:", samples.crs)
print("Kolom:", samples.columns.tolist())
print("Geometri:", samples.geometry.geom_type.value_counts().to_dict())
display(samples.drop(columns="geometry").head())

candidate_cols = [
    "class_name", "kelas", "class", "label", "nama_kelas",
    "tutupan_lahan", "landcover", "jenis", "nama", "kategori", "original_category"
]
label_col = next((c for c in candidate_cols if c in samples.columns), None)
if label_col is None:
    raise ValueError("Kolom label kelas tidak terdeteksi. Kolom tersedia: " + str([c for c in samples.columns if c != samples.geometry.name]))
print("Kolom label terpilih:", label_col)
print(samples[label_col].value_counts(dropna=False))


## 3. Menyamakan label menjadi lima kelas

Label sampel dipetakan ke kelas **Hutan, Perairan, Lahan pertanian, Bangunan, dan Mangrove**. Jika nama kelas dalam GeoJSON berbeda, sesuaikan kamus `CLASS_ALIASES`.

In [ ]:
# Normalisasi label ke lima kelas target
CLASS_ALIASES = {
    "hutan": "Hutan", "forest": "Hutan", "woodland": "Hutan",
    "perairan": "Perairan", "water": "Perairan", "water body": "Perairan",
    "badan air": "Perairan", "sungai": "Perairan", "danau": "Perairan",
    "lahan pertanian": "Lahan pertanian", "pertanian": "Lahan pertanian",
    "agriculture": "Lahan pertanian", "cropland": "Lahan pertanian",
    "sawah": "Lahan pertanian", "perkebunan": "Lahan pertanian",
    "bangunan": "Bangunan", "built-up": "Bangunan", "built up": "Bangunan",
    "building": "Bangunan", "permukiman": "Bangunan", "urban": "Bangunan",
    "mangrove": "Mangrove"
}
TARGET_CLASSES = ["Hutan", "Perairan", "Lahan pertanian", "Bangunan", "Mangrove"]

def normalize_class(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    return CLASS_ALIASES.get(s.casefold(), s)

samples["kelas_rf"] = samples[label_col].apply(normalize_class)
samples = samples[samples["kelas_rf"].isin(TARGET_CLASSES)].copy()
print("Sampel setelah normalisasi:", len(samples))
print(samples["kelas_rf"].value_counts())
missing = sorted(set(TARGET_CLASSES) - set(samples["kelas_rf"].unique()))
if missing:
    raise ValueError(f"Kelas belum ditemukan: {missing}. Sesuaikan CLASS_ALIASES dengan label yang tercetak pada sel sebelumnya.")


## 4. Memeriksa file raster

Pastikan keenam raster hasil clipping tersedia. Raster harus memiliki ukuran, CRS, dan transformasi grid yang sama agar piksel dari setiap band saling cocok.

In [ ]:
missing_files = [str(p) for p in RASTER_FILES.values() if not p.exists()]
if missing_files:
    raise FileNotFoundError("Raster tidak ditemukan:\n" + "\n".join(missing_files))
print("Semua file raster ditemukan.")

## 5. Membaca dan menyusun band

Setiap band dibaca lalu digabungkan menjadi tumpukan data (*stack*) enam band. Piksel bernilai nol atau NoData dikeluarkan dari proses klasifikasi.

In [ ]:
srcs = {b: rasterio.open(p) for b, p in RASTER_FILES.items()}
reference = srcs["B02"]
profile = reference.profile.copy()
height, width = reference.height, reference.width
print("Dimensi:", width, "x", height, "| CRS:", reference.crs, "| bounds:", reference.bounds)

for b, src in srcs.items():
    if (src.width, src.height, src.transform, src.crs) != (reference.width, reference.height, reference.transform, reference.crs):
        raise ValueError(f"Raster {b} tidak berada pada grid yang sama dengan B02.")

stack = np.stack([srcs[b].read(1) for b in BANDS], axis=0)
valid_mask = np.all(np.isfinite(stack), axis=0) & np.all(stack != 0, axis=0)
for i, b in enumerate(BANDS):
    nd = srcs[b].nodata
    if nd is not None:
        valid_mask &= stack[i] != nd
print("Piksel valid:", int(valid_mask.sum()), "/", height * width)

## 6. Mengekstrak nilai band pada sampel

Titik sampel digunakan langsung. Untuk sampel berbentuk poligon, digunakan titik representatif. Nilai enam band pada lokasi tersebut menjadi fitur masukan model.

In [ ]:
# Ekstrak nilai band pada sampel + diagnosis CRS/cakupan
if samples.crs is None:
    raise ValueError("CRS sampel tidak diketahui.")

print("CRS sampel:", samples.crs)
print("CRS raster:", reference.crs)
print("Jumlah fitur sampel:", len(samples))
print("Label sebelum ekstraksi:")
print(samples["kelas_rf"].value_counts(dropna=False))

# Ubah CRS sampel agar sama dengan raster
sample_points = samples.copy()
sample_points["geometry"] = sample_points.geometry.apply(
    lambda geom: geom if geom.geom_type == "Point" else geom.representative_point()
)
sample_points = sample_points.to_crs(reference.crs)

# Cek titik yang jatuh di dalam batas raster
bounds = reference.bounds
inside = sample_points.geometry.apply(
    lambda p: bounds.left <= p.x <= bounds.right and bounds.bottom <= p.y <= bounds.top
)
print("Titik di dalam extent raster:", int(inside.sum()), "/", len(inside))

# Ambil nilai dari keenam band
coords = [(geom.x, geom.y) for geom in sample_points.geometry]
X_all = np.array([
    [next(srcs[b].sample([(x, y)]))[0] for b in BANDS]
    for x, y in coords
], dtype=np.float64)
y_all = samples["kelas_rf"].to_numpy()

print("Contoh nilai band (5 sampel pertama):")
display(pd.DataFrame(X_all[:5], columns=BANDS))
print("Nilai minimum per band:", dict(zip(BANDS, np.nanmin(X_all, axis=0))))
print("Nilai maksimum per band:", dict(zip(BANDS, np.nanmax(X_all, axis=0))))

# Gunakan NoData yang didefinisikan di metadata raster; jangan otomatis membuang nilai 0
good = inside.to_numpy() & np.isfinite(X_all).all(axis=1)
for j, b in enumerate(BANDS):
    nd = srcs[b].nodata
    if nd is not None:
        good &= X_all[:, j] != nd

X = X_all[good]
y = y_all[good]
print("Sampel valid untuk training:", len(y))
print(pd.Series(y).value_counts(dropna=False))

if len(y) == 0:
    raise ValueError(
        "Masih tidak ada sampel valid. Periksa CRS, bounds raster, dan contoh nilai band di atas. "
        "Jika 'Titik di dalam extent raster' = 0, kemungkinan CRS atau cakupan sampel tidak cocok."
    )
if len(np.unique(y)) < 2:
    raise ValueError("Sampel valid hanya memiliki satu kelas; periksa sebaran sampel.")


## 7. Melatih dan mengevaluasi Random Forest

Data dibagi menjadi data latih dan data uji. Model dilatih dengan 300 pohon, lalu dievaluasi menggunakan akurasi, precision, recall, F1-score, dan confusion matrix.

In [ ]:
# Latih dan evaluasi Random Forest
counts = pd.Series(y).value_counts()
stratify_arg = y if counts.min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=stratify_arg
)
model = RandomForestClassifier(
    n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1
)
model.fit(X_train, y_train)
pred_test = model.predict(X_test)
print("Akurasi uji:", round(accuracy_score(y_test, pred_test), 4))
print(classification_report(y_test, pred_test, labels=TARGET_CLASSES, zero_division=0))
display(pd.DataFrame(
    confusion_matrix(y_test, pred_test, labels=TARGET_CLASSES),
    index=TARGET_CLASSES, columns=TARGET_CLASSES
))


## 8. Mengklasifikasikan seluruh piksel

Model terlatih memprediksi kelas pada setiap piksel valid. Prediksi dilakukan per batch agar penggunaan memori lebih terkendali. Nilai 0 menandakan NoData.

In [ ]:
# Prediksi seluruh piksel valid
class_to_id = {name: i + 1 for i, name in enumerate(TARGET_CLASSES)}
id_to_class = {v: k for k, v in class_to_id.items()}
pred_map = np.zeros((height, width), dtype=np.uint8)
rows, cols = np.where(valid_mask)
pixel_matrix = stack[:, rows, cols].T.astype(np.float32)
pred_ids = []
batch_size = 100_000
for start in range(0, len(pixel_matrix), batch_size):
    chunk_pred = model.predict(pixel_matrix[start:start + batch_size])
    pred_ids.extend(class_to_id[str(c)] for c in chunk_pred)
pred_map[rows, cols] = np.asarray(pred_ids, dtype=np.uint8)
print("Piksel terklasifikasi:", int((pred_map > 0).sum()))


## 9. Menampilkan peta klasifikasi dan menyimpan GeoTIFF

Setiap kelas diberi warna yang berbeda. Hasil raster disimpan sebagai `hasil_klasifikasi_jatim.tif` agar bisa digunakan kembali di perangkat lunak GIS atau pemrosesan Python.

In [ ]:
# Peta klasifikasi statis dan simpan GeoTIFF
colors = {
    "Hutan": "#228B22",
    "Perairan": "#1E90FF",
    "Lahan pertanian": "#FFD34E",
    "Bangunan": "#D73027",
    "Mangrove": "#7B3294"
}
cmap = mcolors.ListedColormap([colors[c] for c in TARGET_CLASSES])
norm = mcolors.BoundaryNorm(np.arange(0.5, len(TARGET_CLASSES) + 1.5, 1), cmap.N)
plt.figure(figsize=(12, 8))
plt.imshow(np.ma.masked_where(pred_map == 0, pred_map), cmap=cmap, norm=norm)
cb = plt.colorbar(ticks=np.arange(1, len(TARGET_CLASSES) + 1), shrink=0.75)
cb.ax.set_yticklabels(TARGET_CLASSES)
plt.title("Klasifikasi Tutupan Lahan Jawa Timur — Random Forest")
plt.axis("off")
plt.tight_layout()
plt.show()

out_tif = BASE / "hasil_klasifikasi_jatim.tif"
out_profile = profile.copy()
out_profile.update(count=1, dtype="uint8", nodata=0, compress="lzw")
with rasterio.open(out_tif, "w", **out_profile) as dst:
    dst.write(pred_map, 1)
print("GeoTIFF tersimpan:", out_tif)


## 10. Membuat overlay transparan

Raster klasifikasi diubah menjadi PNG RGBA agar dapat ditumpangkan di atas peta dasar web dengan area NoData tetap transparan.

In [ ]:
# Buat PNG transparan untuk overlay peta web
rgba = np.zeros((height, width, 4), dtype=np.uint8)
hex_to_rgb = {k: tuple(int(v[i:i+2], 16) for i in (1, 3, 5)) for k, v in colors.items()}
for class_name, class_id in class_to_id.items():
    mask = pred_map == class_id
    rgba[mask, :3] = hex_to_rgb[colors[class_name]]
    rgba[mask, 3] = 220
png_path = BASE / "overlay_klasifikasi_jatim.png"
Image.fromarray(rgba, mode="RGBA").save(png_path)
bounds = reference.bounds
image_bounds = [[bounds.bottom, bounds.left], [bounds.top, bounds.right]]
print("Overlay PNG:", png_path)


## 11. Membuat peta interaktif

Peta menggunakan Folium dengan pilihan basemap OpenStreetMap dan citra satelit, overlay hasil klasifikasi, batas wilayah (jika tersedia), legenda, serta kontrol layer. File HTML dapat dibuka di browser.

In [ ]:
# Peta interaktif Folium: basemap, overlay klasifikasi, batas, legenda
center = [(bounds.bottom + bounds.top) / 2, (bounds.left + bounds.right) / 2]
m = folium.Map(location=center, zoom_start=7, tiles=None, control_scale=True)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", overlay=False).add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Citra satelit", overlay=False
).add_to(m)
ImageOverlay(
    name="Klasifikasi tutupan lahan",
    image=str(png_path),
    bounds=image_bounds,
    opacity=0.85,
    interactive=True,
    cross_origin=False,
    zindex=2
).add_to(m)

if BOUNDARY_FILE.exists():
    try:
        boundary = gpd.read_file(BOUNDARY_FILE)
        if boundary.crs is not None:
            boundary = boundary.to_crs("EPSG:4326")
        folium.GeoJson(
            boundary, name="Batas wilayah",
            style_function=lambda feature: {"color": "#111111", "weight": 1.5, "fillOpacity": 0},
            tooltip="Batas wilayah"
        ).add_to(m)
    except Exception as e:
        print("Batas wilayah tidak ditambahkan:", e)

legend_html = '''
<div style="position: fixed; bottom: 35px; left: 35px; z-index: 9999;
background: white; padding: 12px 14px; border: 1px solid #777;
border-radius: 5px; font-size: 13px; box-shadow: 0 1px 5px #777;">
<b>Klasifikasi Tutupan Lahan</b><br>
'''
for name in TARGET_CLASSES:
    legend_html += (
        f'<div style="margin-top:4px;">'
        f'<span style="display:inline-block;width:12px;height:12px;'
        f'background:{colors[name]};margin-right:7px;border:1px solid #555;"></span>'
        f'{name}</div>'
    )
legend_html += '</div>'
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)
html_path = BASE / "peta_klasifikasi_jatim.html"
m.save(str(html_path))
print("Peta interaktif tersimpan:", html_path)
m


## Output
- `peta_klasifikasi_jatim.html`: peta interaktif untuk dibuka di browser.
- `hasil_klasifikasi_jatim.tif`: raster hasil klasifikasi.
- `overlay_klasifikasi_jatim.png`: gambar transparan yang dipakai oleh peta web.

Jika sel normalisasi berhenti karena ada kelas yang tidak ditemukan, cocokkan `CLASS_ALIASES` dengan nilai label yang tampil pada sel pemeriksaan sampel. Kinerja klasifikasi bergantung pada ketepatan label sampel dan cakupan citra.
